# Chapter 3 — Gender and Market Segmentation

Gender-specific catalogue and accord-combination analysis across Men, Women and Unisex classifications. Median segmentation is calculated independently within each classification and analytical level.

Votes indicate platform activity, not sales, market share or unique consumers. Ratings are community evaluations. Catalogue classifications are not evidence of innate preferences.

## 3.1 Configuration and source validation

Raw Fragrantica data supplies ratings, vote counts, classification and ordered accords. Notebook 13's prepared table supplies separately cleaned fragrance and brand names. The join uses URL-derived fragrance_id.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
from pathlib import Path

SOURCE_TABLE = "main.fragrance_analytics.fragrantica_perfumes"
REFERENCE_TABLE = "main.fragrance_analytics.fragrantica_prepared"
OUTPUT_DIR = Path("/Volumes/main/fragrance_analytics/reference_data/chapter_3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENDER_ORDER = ["Men", "Women", "Unisex"]
SEGMENT_ORDER = ["Standout Profiles", "Hidden Favourites", "Widely Explored", "Quiet Profiles"]
MIN_DOMINANT_FRAGRANCES = 100
MIN_DYAD_FRAGRANCES = 100
MIN_TRIAD_FRAGRANCES = 50

df_source = spark.table(SOURCE_TABLE)
df_reference = spark.table(REFERENCE_TABLE)
print("Source rows:", f"{df_source.count():,}")
print("Prepared reference rows:", f"{df_reference.count():,}")
df_source.printSchema()
df_reference.printSchema()

source_id_stats = df_source.agg(F.count("*").alias("rows"), F.countDistinct("id").alias("unique_ids"))
source_id_check = source_id_stats.first()
assert source_id_check["rows"] == source_id_check["unique_ids"], "Raw source must contain one row per id before deduplication."

df_source_keyed = df_source.withColumn(
    "fragrance_id",
    F.regexp_extract("url", r"-([0-9]+)\.html(?:\?.*)?$", 1)
)
assert df_reference.groupBy("fragrance_id").count().filter(F.col("count") > 1).limit(1).count() == 0, "Duplicate reference fragrance_id values."

df_names = df_reference.select(
    "fragrance_id",
    F.col("fragrance_name_clean").alias("fragrance_name"),
    F.col("brand_clean").alias("brand")
)
df_joined = df_source_keyed.join(df_names, "fragrance_id", "left")
source_rows = df_source_keyed.count()
joined_rows = df_joined.count()
assert joined_rows == source_rows, f"Reference join changed row count: {source_rows} -> {joined_rows}"
print("Join rows preserved:", f"{joined_rows:,}")
print("Missing cleaned names:", df_joined.filter(F.col("fragrance_name").isNull() | (F.trim("fragrance_name") == "")).count())
print("Missing cleaned brands:", df_joined.filter(F.col("brand").isNull() | (F.trim("brand") == "")).count())

df_accords = (
    df_joined
    .withColumn("accord_array", F.from_json(
        F.regexp_replace(F.regexp_replace(F.col("main_accords"), "'", '"'), r"\bNone\b", "null"),
        "array<string>"
    ))
    .withColumn("gender_clean", F.initcap(F.lower(F.trim("gender"))))
    .filter(F.col("gender_clean").isin(GENDER_ORDER))
    .withColumn("accord_array", F.expr("filter(transform(accord_array, x -> lower(trim(x))), x -> x is not null and x <> '')"))
    .filter(F.col("id").isNotNull() & (F.size("accord_array") > 0))
    .dropDuplicates(["id"])
)
assert df_accords.groupBy("id").count().filter(F.col("count") > 1).limit(1).count() == 0
print("Classified fragrances with valid accords:", f"{df_accords.count():,}")
display(df_accords.groupBy("gender_clean").agg(F.countDistinct("id").alias("fragrance_count")).orderBy("gender_clean"))

## 3.2 Catalogue distribution by classification

Accord prevalence counts unique fragrance–accord associations across all recorded positions. Fragrances may contribute to multiple accords, so percentages are not additive. No fragrance examples are selected in this section.

In [0]:
gender_distribution_ch3 = (
    df_accords.groupBy("gender_clean")
    .agg(F.countDistinct("id").alias("fragrance_count"))
)
total_gendered_ch3 = gender_distribution_ch3.agg(F.sum("fragrance_count")).first()[0]
gender_distribution_ch3 = (
    gender_distribution_ch3
    .withColumn("catalogue_share_pct", F.round(F.col("fragrance_count") / F.lit(total_gendered_ch3) * 100, 2))
    .orderBy("gender_clean")
)
display(gender_distribution_ch3)

df_all_accords_ch3 = (
    df_accords.select("id", "gender_clean", F.explode("accord_array").alias("accord"))
    .dropDuplicates(["id", "gender_clean", "accord"])
)
gender_denominators_ch3 = df_accords.groupBy("gender_clean").agg(F.countDistinct("id").alias("classification_fragrances"))
accord_prevalence_ch3 = (
    df_all_accords_ch3.groupBy("gender_clean", "accord")
    .agg(F.countDistinct("id").alias("fragrance_count"))
    .join(gender_denominators_ch3, "gender_clean")
    .withColumn("prevalence_pct", F.round(F.col("fragrance_count") / F.col("classification_fragrances") * 100, 2))
    .orderBy("gender_clean", F.desc("prevalence_pct"), "accord")
)
display(accord_prevalence_ch3)

top_accords_ch3 = [
    r["accord"] for r in accord_prevalence_ch3.groupBy("accord")
    .agg(F.sum("fragrance_count").alias("n"))
    .orderBy(F.desc("n"), "accord").limit(30).collect()
]
display(
    accord_prevalence_ch3.filter(F.col("accord").isin(top_accords_ch3))
    .groupBy("accord").pivot("gender_clean", GENDER_ORDER)
    .agg(F.first("prevalence_pct")).orderBy("accord")
)

## 3.3 Shared gender-specific segmentation function

Attention is mean vote count per fragrance; evaluation is mean fragrance rating. Medians are calculated independently within each classification and analytical level. Median ties are assigned to the higher side. Support thresholds are applied before medians are calculated.

In [0]:
def segment_gender_profiles(pdf, entity_col, min_support, level_label):
    required = ["gender_clean", entity_col, "fragrance_count", "mean_votes", "mean_rating"]
    missing = [c for c in required if c not in pdf.columns]
    assert not missing, f"{level_label}: missing columns {missing}"

    data = pdf.dropna(subset=required).copy()
    for col in ["fragrance_count", "mean_votes", "mean_rating"]:
        data[col] = pd.to_numeric(data[col], errors="coerce")
    data = data.dropna(subset=["fragrance_count", "mean_votes", "mean_rating"])
    data = data[data["fragrance_count"] >= min_support].copy()
    if data.empty:
        raise ValueError(f"No eligible {level_label} profiles at support >= {min_support}.")

    data["segment"] = None
    threshold_rows = []
    for gender in GENDER_ORDER:
        mask = data["gender_clean"] == gender
        part = data.loc[mask]
        if part.empty:
            continue
        attention_median = part["mean_votes"].median()
        rating_median = part["mean_rating"].median()
        high_attention = part["mean_votes"] >= attention_median
        high_rating = part["mean_rating"] >= rating_median
        data.loc[mask, "segment"] = np.select(
            [high_attention & high_rating, ~high_attention & high_rating, high_attention & ~high_rating],
            ["Standout Profiles", "Hidden Favourites", "Widely Explored"],
            default="Quiet Profiles"
        )
        threshold_rows.append({
            "classification": gender,
            "analytical_level": level_label,
            "eligible_profile_count": len(part),
            "minimum_fragrance_support": min_support,
            "attention_median": attention_median,
            "rating_median": rating_median
        })

    data["analytical_level"] = level_label
    data["segment"] = pd.Categorical(data["segment"], categories=SEGMENT_ORDER, ordered=True)
    summary = (
        data.groupby(["gender_clean", "segment"], observed=False)
        .agg(
            profile_count=(entity_col, "nunique"),
            fragrance_assignments=("fragrance_count", "sum"),
            mean_fragrances_per_profile=("fragrance_count", "mean"),
            mean_votes_per_profile=("mean_votes", "mean"),
            mean_rating_per_profile=("mean_rating", "mean"),
            median_fragrances_per_profile=("fragrance_count", "median")
        ).reset_index()
    )
    summary["analytical_level"] = level_label
    return data, pd.DataFrame(threshold_rows), summary

## 3.4 Dominant-accord segmentation by classification

The dominant accord is the first valid accord in the original array. Rating and attention metrics use records with a non-null rating and non-negative vote count. Support is the number of distinct fragrances within each classification–accord group.

In [0]:
dominant_profiles_spark_ch3 = (
    df_accords
    .filter(F.col("rating_value").isNotNull())
    .filter(F.col("rating_count").isNotNull() & (F.col("rating_count") >= 0))
    .select("id", "gender_clean", "rating_value", "rating_count", F.element_at("accord_array", 1).alias("dominant_accord"))
    .groupBy("gender_clean", "dominant_accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.avg("rating_count").alias("mean_votes"),
        F.avg("rating_value").alias("mean_rating"),
        F.sum("rating_count").alias("total_votes"),
        F.avg(F.when(F.col("rating_value") >= 4.5, 1.0).otherwise(0.0)).alias("share_rated_4_5_plus")
    )
)
dominant_segmented_ch3, dominant_thresholds_ch3, dominant_summary_ch3 = segment_gender_profiles(
    dominant_profiles_spark_ch3.toPandas(), "dominant_accord",
    MIN_DOMINANT_FRAGRANCES, "Dominant accords"
)
display(dominant_thresholds_ch3)
display(dominant_summary_ch3)
display(dominant_segmented_ch3.sort_values(["gender_clean", "segment", "mean_votes"], ascending=[True, True, False]))

## 3.5 Position-preserving dyad and triad construction

Combinations retain their original accord order. Each fragrance–combination assignment is counted once. Fragrances may contribute to several combinations, so assignment totals are not unique fragrance totals.

In [0]:
df_accord_positions_ch3 = (
    df_accords.select(
        "id", "gender_clean", "rating_value", "rating_count",
        F.posexplode("accord_array").alias("accord_position", "accord")
    ).dropDuplicates(["id", "accord_position"])
)

a, b = df_accord_positions_ch3.alias("a"), df_accord_positions_ch3.alias("b")
df_dyad_assignments_ch3 = (
    a.join(b, (F.col("a.id") == F.col("b.id")) & (F.col("a.accord_position") < F.col("b.accord_position")), "inner")
    .select(
        F.col("a.id").alias("id"), F.col("a.gender_clean").alias("gender_clean"),
        F.col("a.rating_value").alias("rating_value"), F.col("a.rating_count").alias("rating_count"),
        F.concat_ws(" + ", F.col("a.accord"), F.col("b.accord")).alias("dyad")
    ).dropDuplicates(["id", "dyad"])
)

a, b, c = df_accord_positions_ch3.alias("a"), df_accord_positions_ch3.alias("b"), df_accord_positions_ch3.alias("c")
df_triad_assignments_ch3 = (
    a.join(b, (F.col("a.id") == F.col("b.id")) & (F.col("a.accord_position") < F.col("b.accord_position")), "inner")
    .join(c, (F.col("a.id") == F.col("c.id")) & (F.col("b.id") == F.col("c.id")) & (F.col("b.accord_position") < F.col("c.accord_position")), "inner")
    .select(
        F.col("a.id").alias("id"), F.col("a.gender_clean").alias("gender_clean"),
        F.col("a.rating_value").alias("rating_value"), F.col("a.rating_count").alias("rating_count"),
        F.concat_ws(" + ", F.col("a.accord"), F.col("b.accord"), F.col("c.accord")).alias("triad")
    ).dropDuplicates(["id", "triad"])
)

def aggregate_combinations(df, entity_col):
    return (
        df.filter(F.col("rating_value").isNotNull())
        .filter(F.col("rating_count").isNotNull() & (F.col("rating_count") >= 0))
        .groupBy("gender_clean", entity_col)
        .agg(
            F.countDistinct("id").alias("fragrance_count"),
            F.avg("rating_count").alias("mean_votes"),
            F.avg("rating_value").alias("mean_rating"),
            F.sum("rating_count").alias("total_votes"),
            F.avg(F.when(F.col("rating_value") >= 4.5, 1.0).otherwise(0.0)).alias("share_rated_4_5_plus")
        )
    )

dyad_profiles_spark_ch3 = aggregate_combinations(df_dyad_assignments_ch3, "dyad")
triad_profiles_spark_ch3 = aggregate_combinations(df_triad_assignments_ch3, "triad")

print("Fragrance–dyad assignments:", f"{df_dyad_assignments_ch3.count():,}")
print("Fragrance–triad assignments:", f"{df_triad_assignments_ch3.count():,}")
display(dyad_profiles_spark_ch3.groupBy("gender_clean").count().orderBy("gender_clean"))
display(triad_profiles_spark_ch3.groupBy("gender_clean").count().orderBy("gender_clean"))

## 3.6 Dyad and triad segmentation

Dyads and triads are segmented independently within each classification. Support thresholds are applied before classification-specific medians are calculated.

In [0]:
dyad_segmented_ch3, dyad_thresholds_ch3, dyad_summary_ch3 = segment_gender_profiles(
    dyad_profiles_spark_ch3.toPandas(), "dyad", MIN_DYAD_FRAGRANCES, "Dyads"
)
triad_segmented_ch3, triad_thresholds_ch3, triad_summary_ch3 = segment_gender_profiles(
    triad_profiles_spark_ch3.toPandas(), "triad", MIN_TRIAD_FRAGRANCES, "Triads"
)

gender_thresholds_ch3 = pd.concat(
    [dominant_thresholds_ch3, dyad_thresholds_ch3, triad_thresholds_ch3],
    ignore_index=True
)
gender_segment_summary_ch3 = pd.concat(
    [dominant_summary_ch3, dyad_summary_ch3, triad_summary_ch3],
    ignore_index=True
)
display(gender_thresholds_ch3)
display(gender_segment_summary_ch3.sort_values(["analytical_level", "gender_clean", "segment"]))
display(dyad_segmented_ch3.sort_values(["gender_clean", "segment", "mean_votes"], ascending=[True, True, False]))
display(triad_segmented_ch3.sort_values(["gender_clean", "segment", "mean_votes"], ascending=[True, True, False]))

## 3.7 Top-three fragrance examples within each profile

For each eligible dominant accord, dyad and triad, the three fragrances with the highest vote counts are selected within the same classification, profile and segment. Third-place ties are audited. A deterministic secondary ordering retains exactly three examples when at least three candidates exist. The full ranked candidate set is retained for further review.

In [0]:
fragrance_lookup_ch3 = (
    df_accords.select(
        "id", "gender_clean", "fragrance_name", "brand",
        F.col("rating_value").cast("double").alias("rating"),
        F.col("rating_count").cast("long").alias("rating_count")
    ).dropDuplicates(["id"])
)

def rank_profile_fragrances(profile_pdf, entity_col, assignment_df, level_label):
    assignments = assignment_df.select("id", "gender_clean", entity_col).dropDuplicates()
    candidates = assignments.join(
        fragrance_lookup_ch3, ["id", "gender_clean"], "left"
    ).toPandas()

    profile_labels = profile_pdf[["gender_clean", entity_col, "segment"]].drop_duplicates()
    candidates = candidates.merge(
        profile_labels,
        on=["gender_clean", entity_col],
        how="inner",
        validate="many_to_one"
    )

    candidates["rating_count"] = pd.to_numeric(candidates["rating_count"], errors="coerce").fillna(-1)
    candidates["rating"] = pd.to_numeric(candidates["rating"], errors="coerce")
    candidates["brand"] = candidates["brand"].fillna("[Brand unavailable]")
    candidates["fragrance_name"] = candidates["fragrance_name"].fillna("[Fragrance name unavailable]")

    group_cols = ["gender_clean", "segment", entity_col]
    candidates = candidates.sort_values(
        group_cols + ["rating_count", "rating", "brand", "fragrance_name", "id"],
        ascending=[True, True, True, False, False, True, True, True],
        kind="mergesort"
    )
    candidates["rank_by_votes"] = candidates.groupby(group_cols).cumcount() + 1
    candidates["third_place_vote_count"] = candidates.groupby(group_cols)["rating_count"].transform(
        lambda values: values.nlargest(3).iloc[-1] if len(values) >= 3 else np.nan
    )
    candidates["third_place_tie_count"] = candidates.groupby(group_cols)["rating_count"].transform(
        lambda values: int((values == values.nlargest(3).iloc[-1]).sum()) if len(values) >= 3 else 0
    )
    candidates["analytical_level"] = level_label
    candidates["classification"] = candidates["gender_clean"]
    return candidates

dominant_assignments_ch3 = (
    df_accords.filter(F.col("rating_value").isNotNull()).filter(F.col("rating_count").isNotNull() & (F.col("rating_count") >= 0)).select(
        "id", "gender_clean",
        F.element_at("accord_array", 1).alias("dominant_accord")
    ).dropDuplicates(["id", "gender_clean", "dominant_accord"])
)

dominant_candidates_ch3 = rank_profile_fragrances(
    dominant_segmented_ch3, "dominant_accord", dominant_assignments_ch3, "Dominant accords"
)
dyad_candidates_ch3 = rank_profile_fragrances(
    dyad_segmented_ch3, "dyad", df_dyad_assignments_ch3, "Dyads"
)
triad_candidates_ch3 = rank_profile_fragrances(
    triad_segmented_ch3, "triad", df_triad_assignments_ch3, "Triads"
)

all_example_candidates_ch3 = pd.concat(
    [dominant_candidates_ch3, dyad_candidates_ch3, triad_candidates_ch3],
    ignore_index=True
)
top_three_examples_ch3 = all_example_candidates_ch3[
    all_example_candidates_ch3["rank_by_votes"] <= 3
].copy()

example_columns_ch3 = [
    "analytical_level", "classification", "segment", "rank_by_votes",
    "dominant_accord", "dyad", "triad", "fragrance_name", "brand",
    "rating", "rating_count", "id", "third_place_vote_count", "third_place_tie_count"
]
example_columns_ch3 = [c for c in example_columns_ch3 if c in top_three_examples_ch3.columns]
display(top_three_examples_ch3[example_columns_ch3].sort_values(
    ["analytical_level", "classification", "segment", "rank_by_votes"]
))

tie_audit_ch3 = (
    all_example_candidates_ch3.groupby(
        ["analytical_level", "classification", "segment", "dominant_accord", "dyad", "triad"],
        dropna=False
    ).agg(
        candidate_count=("id", "count"),
        third_place_vote_count=("third_place_vote_count", "first"),
        third_place_tie_count=("third_place_tie_count", "first")
    ).reset_index()
)
display(tie_audit_ch3)

## 3.8 Cross-level comparison

Counts refer to eligible profiles after support thresholds. Fragrance assignments summed across profiles are not unique catalogue totals. Segment averages are unweighted means across profiles, not fragrance-weighted averages.

In [0]:
gender_level_comparison_ch3 = (
    gender_segment_summary_ch3.groupby(["gender_clean", "analytical_level"], as_index=False)
    .agg(
        eligible_profile_count=("profile_count", "sum"),
        fragrance_assignments=("fragrance_assignments", "sum"),
        mean_votes_per_profile=("mean_votes_per_profile", "mean"),
        mean_rating_per_profile=("mean_rating_per_profile", "mean")
    )
)
display(gender_level_comparison_ch3.sort_values(["gender_clean", "analytical_level"]))

gender_segment_comparison_ch3 = gender_segment_summary_ch3.pivot_table(
    index=["analytical_level", "segment"],
    columns="gender_clean",
    values="profile_count",
    aggfunc="sum"
).reindex(columns=GENDER_ORDER).reset_index()
display(gender_segment_comparison_ch3)

## 3.9 Export analytical reference tables

Complete analytical tables are exported for manuscript analysis and reproducibility. The source and prepared reference tables are not overwritten.

In [0]:
exports_ch3 = {
    "chapter_3_gender_distribution.csv": gender_distribution_ch3.toPandas(),
    "chapter_3_accord_prevalence.csv": accord_prevalence_ch3.toPandas(),
    "chapter_3_gender_thresholds.csv": gender_thresholds_ch3,
    "chapter_3_gender_segment_summary.csv": gender_segment_summary_ch3,
    "chapter_3_dominant_accord_profiles.csv": dominant_segmented_ch3,
    "chapter_3_dyad_profiles.csv": dyad_segmented_ch3,
    "chapter_3_triad_profiles.csv": triad_segmented_ch3,
    "chapter_3_top_three_fragrance_examples.csv": top_three_examples_ch3,
    "chapter_3_all_ranked_example_candidates.csv": all_example_candidates_ch3,
    "chapter_3_example_tie_audit.csv": tie_audit_ch3,
    "chapter_3_gender_level_comparison.csv": gender_level_comparison_ch3,
    "chapter_3_gender_segment_comparison.csv": gender_segment_comparison_ch3
}
for filename, export_df in exports_ch3.items():
    output_path = OUTPUT_DIR / filename
    export_df.to_csv(output_path, index=False)
    assert output_path.is_file() and output_path.stat().st_size > 0, f"Export failed: {output_path}"
    print(f"Saved: {output_path} ({output_path.stat().st_size:,} bytes)")
print("Verified Chapter 3 exports:", len(exports_ch3))

## Reproducibility notes

- Source: main.fragrance_analytics.fragrantica_perfumes.
- Clean fragrance and brand names: main.fragrance_analytics.fragrantica_prepared, joined by URL-derived fragrance_id.
- Source records are validated at one row per Fragrantica id; the reference join must preserve row count.
- Accord positions are retained. The first accord is dominant; dyads and triads are ordered combinations.
- Minimum support: 100 fragrances for dominant accords, 100 for dyads and 50 for triads.
- Medians are calculated independently by classification and analytical level; median ties are assigned to the higher side.
- Top-three examples are selected by vote count within classification–profile–segment; full ranked candidates and third-place ties are retained.
- Vote counts are platform activity, not sales, market share or unique consumers. Ratings are platform evaluations.
- Fragrance assignments across profiles are not unique catalogue totals.
- This notebook produces analytical reference tables. Publication figures should follow the established Chapter 1 and Chapter 2 visual conventions.